            # BirdCLEF 2026 — Submission Notebook

            **Study:** `study_20260414_211157_full_dataset_v4`
            **Experiment:** `exp_001`
**Best F1:** 0.5858
            **Generated by:** autonomous research agent

            This notebook is **self-contained** — no local imports needed.
            It loads pre-trained model weights, processes test soundscapes,
            and writes `submission.csv`. Runs on CPU within 90 minutes.

            ## Setup

            Before submitting to Kaggle:
            1. Upload `best_model.pt` as a Kaggle dataset (e.g., `your-username/birdclef-model`)
            2. Attach that dataset to this notebook
            3. Update `WEIGHTS_PATH` below to match the dataset path


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# === Force CPU-only execution ===
os.environ["CUDA_VISIBLE_DEVICES"] = ""
device = torch.device("cpu")
torch.set_num_threads(os.cpu_count() or 4)

# === Paths — update WEIGHTS_PATH for your Kaggle dataset ===
# On Kaggle, attached datasets are at /kaggle/input/<dataset-name>/
WEIGHTS_PATH = "/kaggle/input/birdclef-model/best_model.pt"
THRESHOLDS_PATH = "/kaggle/input/birdclef-model/best_thresholds.npy"
TEST_SOUNDSCAPES = "/kaggle/input/birdclef-2026/test_soundscapes"
SAMPLE_SUBMISSION = "/kaggle/input/birdclef-2026/sample_submission.csv"

# === Audio preprocessing config (must match training) ===
SAMPLE_RATE = 32000
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 20.0
WINDOW_SECONDS = 5.0
TOP_DB = 80.0

# === Model config ===
NUM_CLASSES = 206
BACKBONE = "efficientnet_b0"

# === Class mapping: model output index -> species ID ===
CLASS_IDS = ["1161364", "116570", "1176823", "1595929", "209233", "22930", "22956", "22961", "22967", "22973", "22983", "22985", "23150", "23154", "23158", "23176", "23724", "24279", "24285", "24287", "24321", "244024", "25092", "25214", "326272", "41970", "43435", "47144", "476521", "516975", "555123", "555145", "555146", "64898", "65377", "65380", "66971", "67107", "67252", "70711", "738183", "74113", "74580", "760266", "ashgre1", "astcra1", "bafcur1", "baffal1", "banana", "barant1", "batbel1", "baymac", "bbwduc", "bcwfin2", "bkcdon", "bkhpar", "blchaw1", "blheag1", "blttit1", "bncfly", "bobfly1", "brcmar1", "brnowl", "bucmot4", "bucpar", "bufpar", "bunibi1", "burowl", "camfli1", "chacha1", "chbmoc1", "chobla1", "chvcon1", "cibspi1", "coffal1", "compau", "compot1", "crbthr1", "crebec1", "dwatin1", "epaori4", "eulfly1", "fabwre1", "fepowl", "ficman1", "flawar1", "fotfly", "fusfly1", "gilhum1", "giwrai1", "glteme1", "grasal3", "greani1", "greant1", "greela", "grekis", "grepot1", "gretho2", "greyel", "grfdov1", "grhtan1", "gycwor1", "horscr1", "houspa", "hyamac1", "larela1", "lesela1", "lesgrf1", "limpki", "linwoo1", "litcuc2", "litnig1", "mabpar", "magant1", "magtan2", "masgna1", "nacnig1", "ocecra1", "oliwoo1", "orbtro3", "orwpar", "osprey", "pabspi1", "palhor3", "paltan1", "phecuc1", "picpig2", "pirfly1", "plasla1", "platyr1", "plcjay1", "pluibi1", "purjay1", "pvttyr1", "ragmac1", "rebscy1", "recfin1", "redjun", "relser1", "rinkin1", "rivwar1", "roahaw", "rubthr1", "rufcac2", "rufcas2", "rufgna3", "rufhor2", "rufnig1", "ruftho1", "ruftof1", "rumfly1", "ruther1", "rutjac1", "sabspa1", "saffin", "saytan1", "scadov1", "schpar1", "scther1", "shcfly1", "shshaw", "shtnig1", "sibtan2", "smbani", "smbtin1", "sobcac1", "sobtyr1", "socfly1", "sofspi1", "souant1", "soulap1", "souscr1", "spbant3", "spispi1", "sptnig1", "squcuc1", "stbwoo2", "strcuc1", "strher2", "strowl1", "swthum1", "swtman1", "tattin1", "thlwre1", "toctou1", "trokin", "trsowl", "undtin1", "varant1", "watjac1", "wesfie1", "wfwduc1", "whbant2", "whbwar2", "whiwoo1", "whlspi1", "whnjay1", "whtdov", "whwpic1", "y00678", "yebcar", "yebela1", "yecmac", "yecpar", "yehcar1", "yeofly1"]

print(f"Device: {device}")
print(f"Weights: {WEIGHTS_PATH}")
print(f"Test dir: {TEST_SOUNDSCAPES}")
print(f"Classes: {NUM_CLASSES} (trained) / 234 (submission)")


## Model architecture

TorchvisionAdapter wrapping `efficientnet_b0` — defined inline, no local imports needed.

In [ ]:
import torchvision.models as tvm

class TorchvisionAdapter(nn.Module):
    """Wraps a torchvision backbone for single-channel spectrograms.

    Input: (B, 1, 128, 313) -> resize to (B, 3, 224, 224) -> backbone -> head.
    """
    def __init__(self, backbone_name, num_classes, pretrained=False):
        super().__init__()
        ctor = getattr(tvm, backbone_name)
        weights = None
        if pretrained:
            weights_cls_name = "".join(
                p.capitalize() for p in backbone_name.split("_")
            ) + "_Weights"
            weights_cls = getattr(tvm, weights_cls_name, None)
            if weights_cls is not None:
                weights = getattr(weights_cls, "DEFAULT", None)
        try:
            backbone = ctor(weights=weights)
        except Exception:
            backbone = ctor(weights=None)

        # Replace classifier head with Identity
        if hasattr(backbone, "fc") and isinstance(backbone.fc, nn.Module):
            backbone.fc = nn.Identity()
        elif hasattr(backbone, "classifier") and isinstance(backbone.classifier, nn.Module):
            backbone.classifier = nn.Identity()

        self.backbone = backbone
        self.head = nn.LazyLinear(num_classes)
        self.input_size = (224, 224)

    def forward(self, x):
        if x.size(1) == 1:
            x = x.expand(-1, 3, -1, -1)
        if x.shape[-2:] != self.input_size:
            x = F.interpolate(x, size=self.input_size, mode="bilinear", align_corners=False)
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.flatten(1)
        return self.head(features)

# Instantiate and load weights
model = TorchvisionAdapter(BACKBONE, NUM_CLASSES)
model = model.to(device)

# Initialize lazy layers with a dummy forward pass
with torch.no_grad():
    dummy = torch.zeros(1, 1, N_MELS, 313, device=device)
    model(dummy)

# Load trained weights
if os.path.exists(WEIGHTS_PATH):
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    print(f"Loaded weights from {WEIGHTS_PATH}")
else:
    print(f"WARNING: {WEIGHTS_PATH} not found! Using random weights.")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {BACKBONE} with {n_params:,} parameters")


## Audio preprocessing

Converts raw audio to log-mel-spectrograms using the same parameters as training.

In [ ]:
import librosa

def audio_to_spectrograms(audio_path, sr=SAMPLE_RATE):
    """Load audio and split into 5-second mel-spectrogram windows.

    Returns list of (window_end_seconds, spectrogram) tuples.
    """
    y, _ = librosa.load(str(audio_path), sr=sr, mono=True)
    y = y.astype(np.float32)

    window_samples = int(sr * WINDOW_SECONDS)
    specs = []
    start = 0
    window_idx = 0

    while start + window_samples <= len(y):
        chunk = y[start : start + window_samples]
        mel = librosa.feature.melspectrogram(
            y=chunk, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, fmin=FMIN, fmax=sr / 2, power=2.0,
        )
        mel_db = librosa.power_to_db(mel, top_db=TOP_DB).astype(np.float32)
        end_seconds = (start + window_samples) // sr * WINDOW_SECONDS
        # BirdCLEF row_id uses end time: 5, 10, 15, ...
        end_sec = int((window_idx + 1) * WINDOW_SECONDS)
        specs.append((end_sec, mel_db))
        start += window_samples
        window_idx += 1

    # Handle remaining audio (pad if needed)
    if start < len(y) and len(y) - start > sr:  # at least 1 second
        chunk = np.zeros(window_samples, dtype=np.float32)
        remaining = y[start:]
        chunk[:len(remaining)] = remaining
        mel = librosa.feature.melspectrogram(
            y=chunk, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, fmin=FMIN, fmax=sr / 2, power=2.0,
        )
        mel_db = librosa.power_to_db(mel, top_db=TOP_DB).astype(np.float32)
        end_sec = int((window_idx + 1) * WINDOW_SECONDS)
        specs.append((end_sec, mel_db))

    return specs

print("Audio preprocessing ready")


## Inference on test soundscapes

Process each test soundscape, predict species probabilities using optimized per-class thresholds, and build the submission DataFrame.

In [ ]:
from pathlib import Path
import glob

# Full species list for submission (234 species)
ALL_SPECIES = ["1161364", "116570", "1176823", "1491113", "1595929", "209233", "22930", "22956", "22961", "22967", "22973", "22983", "22985", "23150", "23154", "23158", "23176", "23724", "24279", "24285", "24287", "24321", "244024", "25073", "25092", "25214", "326272", "41970", "43435", "47144", "47158son01", "47158son02", "47158son03", "47158son04", "47158son05", "47158son06", "47158son07", "47158son08", "47158son09", "47158son10", "47158son11", "47158son12", "47158son13", "47158son14", "47158son15", "47158son16", "47158son17", "47158son18", "47158son19", "47158son20", "47158son21", "47158son22", "47158son23", "47158son24", "47158son25", "476521", "516975", "517063", "555123", "555145", "555146", "64898", "65377", "65380", "66971", "67107", "67252", "70711", "738183", "74113", "74580", "760266", "ashgre1", "astcra1", "bafcur1", "baffal1", "banana", "barant1", "batbel1", "baymac", "bbwduc", "bcwfin2", "bkcdon", "bkhpar", "blchaw1", "blheag1", "blttit1", "bncfly", "bobfly1", "brcmar1", "brnowl", "bucmot4", "bucpar", "bufpar", "bunibi1", "burowl", "camfli1", "chacha1", "chbmoc1", "chobla1", "chvcon1", "cibspi1", "coffal1", "compau", "compot1", "crbthr1", "crebec1", "dwatin1", "epaori4", "eulfly1", "fabwre1", "fepowl", "ficman1", "flawar1", "fotfly", "fusfly1", "gilhum1", "giwrai1", "glteme1", "grasal3", "greani1", "greant1", "greela", "grekis", "grepot1", "gretho2", "greyel", "grfdov1", "grhtan1", "gycwor1", "horscr1", "houspa", "hyamac1", "larela1", "lesela1", "lesgrf1", "limpki", "linwoo1", "litcuc2", "litnig1", "mabpar", "magant1", "magtan2", "masgna1", "nacnig1", "ocecra1", "oliwoo1", "orbtro3", "orwpar", "osprey", "pabspi1", "palhor3", "paltan1", "phecuc1", "picpig2", "pirfly1", "plasla1", "platyr1", "plcjay1", "pluibi1", "purjay1", "pvttyr1", "ragmac1", "rebscy1", "recfin1", "redjun", "relser1", "rinkin1", "rivwar1", "roahaw", "rubthr1", "rufcac2", "rufcas2", "rufgna3", "rufhor2", "rufnig1", "ruftho1", "ruftof1", "rumfly1", "ruther1", "rutjac1", "sabspa1", "saffin", "saytan1", "scadov1", "schpar1", "scther1", "shcfly1", "shshaw", "shtnig1", "sibtan2", "smbani", "smbtin1", "sobcac1", "sobtyr1", "socfly1", "sofspi1", "souant1", "soulap1", "souscr1", "spbant3", "spispi1", "sptnig1", "squcuc1", "stbwoo2", "strcuc1", "strher2", "strowl1", "swthum1", "swtman1", "tattin1", "thlwre1", "toctou1", "trokin", "trsowl", "undtin1", "varant1", "watjac1", "wesfie1", "wfwduc1", "whbant2", "whbwar2", "whiwoo1", "whlspi1", "whnjay1", "whtdov", "whwpic1", "y00678", "yebcar", "yebela1", "yecmac", "yecpar", "yehcar1", "yeofly1"]

# NOTE: Competition metric is macro-averaged ROC-AUC, so we output
# raw probabilities, NOT binary predictions. Thresholds are not needed
# for submission but kept for local F1 evaluation.
print("Submission mode: outputting raw probabilities (ROC-AUC metric)")

# Map our model's class indices to submission column positions
# Our model outputs NUM_CLASSES probabilities in CLASS_IDS order
# Submission needs 234 columns in ALL_SPECIES order
model_idx_to_sub_col = {}
for model_idx, cid in enumerate(CLASS_IDS):
    if cid in ALL_SPECIES:
        sub_col = ALL_SPECIES.index(cid)
        model_idx_to_sub_col[model_idx] = sub_col

print(f"Mapped {len(model_idx_to_sub_col)}/{NUM_CLASSES} model classes to submission columns")

# Read sample submission to get the expected row_ids
sample_sub = pd.read_csv(SAMPLE_SUBMISSION)
expected_row_ids = set(sample_sub["row_id"].tolist())
print(f"Expected {len(expected_row_ids)} rows in submission")

# Process test soundscapes
test_dir = Path(TEST_SOUNDSCAPES)
audio_files = sorted(glob.glob(str(test_dir / "*.ogg")))
if not audio_files:
    audio_files = sorted(glob.glob(str(test_dir / "*.wav")))
if not audio_files:
    audio_files = sorted(glob.glob(str(test_dir / "*.flac")))

print(f"Found {len(audio_files)} test soundscape files")

rows = []
start_time = time.time()

for file_idx, audio_path in enumerate(audio_files):
    filename = Path(audio_path).stem  # e.g. BC2026_Test_0001_S05_20250227_010002
    specs = audio_to_spectrograms(audio_path)

    for end_sec, mel_db in specs:
        row_id = f"{filename}_{end_sec}"

        # Only predict for row_ids that are in the sample submission
        if row_id not in expected_row_ids:
            continue

        # Prepare input tensor: (1, 1, n_mels, time)
        spec_tensor = torch.from_numpy(mel_db).unsqueeze(0).unsqueeze(0).to(device)

        with torch.inference_mode():
            logits = model(spec_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()[0]

        # Output raw probabilities (competition metric is ROC-AUC)
        # Species our model was not trained on get 0.0
        row = np.zeros(len(ALL_SPECIES), dtype=np.float32)
        for model_idx, sub_col in model_idx_to_sub_col.items():
            row[sub_col] = float(probs[model_idx])

        rows.append({"row_id": row_id, **dict(zip(ALL_SPECIES, row))})

    if (file_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"  Processed {file_idx + 1}/{len(audio_files)} files ({elapsed:.0f}s)")

elapsed = time.time() - start_time
print(f"Inference complete: {len(rows)} predictions in {elapsed:.0f}s")


## Write submission.csv

In [ ]:
# Build submission DataFrame
submission = pd.DataFrame(rows)

# Ensure all expected row_ids are present (fill missing with uniform prior)
if len(submission) < len(sample_sub):
    missing_ids = set(sample_sub["row_id"]) - set(submission["row_id"])
    if missing_ids:
        print(f"WARNING: {len(missing_ids)} missing row_ids, filling with uniform prior")
        uniform = 1.0 / len(ALL_SPECIES)
        for rid in missing_ids:
            row = {"row_id": rid}
            row.update({sp: uniform for sp in ALL_SPECIES})
            submission = pd.concat([submission, pd.DataFrame([row])], ignore_index=True)

# Ensure correct column order
submission = submission[["row_id"] + ALL_SPECIES]

# Save
submission.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv: {submission.shape[0]} rows x {submission.shape[1]} columns")
print(submission.head())
